# Capítulo 6 — Reprodutibilidade: transform, divisão repetível, feature store e versões

## Missão
Capítulo 6, padrões 21 a 27. É o que separa um notebook de um sistema. Você vai:
- provocar e corrigir o **desvio treino/serving** com o padrão *Transform* (a preparação salva junto com o modelo);
- fazer uma **divisão repetível** por hash do cliente;
- usar uma **ponte de esquema** para uma coluna nova;
- calcular uma **inferência com janela** igual no treino e no serving;
- montar um **pipeline de workflow** com cache;
- usar uma **feature store** com junção no ponto do tempo;
- **versionar** os modelos.

**No fim deste caderno você terá:** o módulo `meu_mlops/reprodutivel.py` (divisão por hash, feature de janela, runner de workflow com cache, mini feature store e registro de versões).

**Leitura essencial (opcional, livro em inglês):**
- Design Pattern 21: Transform · PDF p. 301
- Design Pattern 22: Repeatable Splitting · PDF p. 308
- Design Pattern 23: Bridged Schema · PDF p. 315
- Design Pattern 24: Windowed Inference · PDF p. 322
- Design Pattern 25: Workflow Pipeline · PDF p. 331
- Design Pattern 26: Feature Store · PDF p. 341
- Design Pattern 27: Model Versioning · PDF p. 357

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap06_reprodutibilidade_transform_divisao_repetivel.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_mlops/
os.makedirs("../meu_mlops", exist_ok=True)
open("../meu_mlops/__init__.py", "a").close()        # meu_mlops/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_mlops")

## Ideia em 30 segundos
- **Desvio treino/serving:** é o bug mais caro de ML. O treino normaliza com a média do treino e o serving esquece, ou usa outra média.
- **Transform:** guarda a preparação **dentro** do artefato do modelo.
- **Divisão repetível:** `hash(chave) % 100 < 80` manda sempre o mesmo cliente para o mesmo lado, mesmo com dados novos.
- **Feature store:** uma única definição de cada feature, servida *offline* (treino, com junção no ponto do tempo) e *online* (serving, o valor mais recente).

### 6.1 **As peças.** Crie `reprodutivel.py` com:
- `lado_por_hash(chave, pct_treino=80)`;
- `pedidos_ultima_hora(df)`: para cada pedido, quantos pedidos o **mesmo restaurante** recebeu nos 60 minutos **anteriores** (`rolling('60min')` por restaurante, sem contar o próprio);
- `Workflow`: `passo(nome, funcao, *dependencias)` registra e `rodar()` executa em ordem, guardando o resultado de cada passo em `../saidas/cache/<nome>.joblib`. Se o cache existe, pula;
- `FeatureStore`: `offline(entidades_com_tempo)`, com `merge_asof` do valor válido **antes** de cada momento, e `online(id)`, com o último valor;
- `Registro(pasta)`: `salvar(modelo, nome, metricas)` cria `v1`, `v2`... com o `metadados.json`, e `carregar(nome, versao=None)` pega a mais recente.

_A célula grava `meu_mlops/reprodutivel.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_mlops/reprodutivel.py


### 6.2 **Transform: o desvio treino/serving.** Treine uma `Ridge` com as numéricas **padronizadas à mão** (média e desvio do treino). No "serving", o engenheiro esquece de padronizar: compare o MAE. Depois empacote `StandardScaler` + `Ridge` num `make_pipeline`, salve com `joblib` e carregue: a preparação viaja junto, e o erro some.

### 6.3 **Divisão repetível.** Separe os clientes por `lado_por_hash(cliente_id)`. Mostre que nenhum cliente aparece nos dois lados (o que acontece com a divisão aleatória por **pedido**: o modelo "conhece" o cliente). Depois acrescente 10 mil pedidos novos e confira que os clientes antigos **não mudam de lado**.

### 6.4 **Ponte de esquema e janela.**
(a) A partir de junho, o app passou a registrar a `gorjeta`. Para usar os dados antigos, imite a distribuição: preencha a gorjeta dos pedidos antigos sorteando dos valores recentes (*bridged schema*) e compare com zerar.
(b) Calcule `pedidos_ultima_hora` (a carga do restaurante) e veja quanto essa feature de janela melhora o MAE de um `HistGradientBoostingRegressor`.

### 6.5 **Workflow com cache e feature store.**
(a) Monte um `Workflow` com três passos: `dados` → `features` → `modelo`. Rode duas vezes: na segunda, tudo vem do cache. Apague o cache do passo `modelo` e rode de novo: só ele reexecuta.
(b) Crie uma feature store com a **nota média acumulada** de cada restaurante ao longo do tempo. Faça a junção *offline* correta no tempo (`merge_asof`) e compare com o vazamento de usar a média do ano inteiro.

### 6.6 **Versionar modelos.** Com um `Registro("../saidas/registro")` (limpe a pasta antes), salve o Ridge como `tempo_entrega` v1 e o boosting com a carga do restaurante como v2, cada um com o MAE nos metadados. Carregue a versão mais recente e a v1 explicitamente: são os dois modelos que um teste A/B serviria lado a lado.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
